In [3]:

import warnings
warnings.filterwarnings("ignore")
import sys
import os
from arch import arch_model
import numpy as np 
import pandas as pd
import torch
import math
from statsmodels.tsa.stattools import adfuller
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, r2_score
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import norm
from scipy import stats
import scipy.stats as scipy_stats
import plotly.graph_objects as go
from pathlib import Path

In [4]:
import importlib

workspace_root = Path.cwd()
if not (workspace_root / "ultility").exists():
    workspace_root = workspace_root.parent
for p in [workspace_root, workspace_root / "model"]:
    if p.exists():
        p_str = str(p)
        if p_str not in sys.path:
            sys.path.insert(0, p_str)

import ultility.data_loader
import ultility.metrics
import ultility.models_garch
import ultility.models_transformer
import train_pipeline
import ultility.var_calculator as var_calculator
import ultility.transformer_garch
importlib.reload(ultility.data_loader)
importlib.reload(ultility.metrics)
importlib.reload(ultility.models_garch)
importlib.reload(ultility.models_transformer)
importlib.reload(train_pipeline)
importlib.reload(ultility.var_calculator)

<module 'ultility.var_calculator' from 'd:\\UIT_LEARNING_MATERIAL\\07.Research\\code\\ultility\\var_calculator.py'>

In [13]:
for csv_name, key in [('VN30_INDEX.csv', 'VN30 Index'), ('VN_INDEX.csv', 'VN Index')]:
    df_temp = pd.read_csv(f'../dataset/{csv_name}')
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=True, errors='coerce')
    df_temp = df_temp.sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{key}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[key] = df_temp_filtered.set_index('time')[ret_col] * 100

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f'../dataset/{file}')
    if 'Date' in df_temp.columns:
        df_temp.rename(columns={'Date': 'time'}, inplace=True)
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=False, errors='coerce')
    df_temp = df_temp.dropna(subset=['time']).sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{file}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[file.replace('.csv', '')] = df_temp_filtered.set_index('time')[ret_col] * 100


In [14]:
summary_split = pd.DataFrame({
    "Dataset": [
        "VN30 Index", "VN Index", "DAX_40", "EuroNext_100", "IBEX_35", "KOSPI_index", "SMI", "snp500", "Nikkei_225"
    ],
    "train_size": [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "val_size":   [1096, 1103, 1104, 1115, 1362, 1109, 1135, 1109, 1174],
    "test_size":  [925, 925, 972, 979, 923, 881, 901, 927, 1062],
    "split_i":    [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "split_j":    [3067, 3067, 3087, 3120, 3177, 3053, 3122, 3097, 2851]
})
split_df = summary_split.set_index("Dataset")[["train_size", "val_size", "test_size"]]
split_df.columns = ["Train", "Val", "Test"]
split_df

,Train,Val,Test
Dataset,,,
VN30 Index,1971,1096,925
VN Index,1964,1103,925
DAX_40,1983,1104,972
EuroNext_100,2005,1115,979
IBEX_35,1815,1362,923
KOSPI_index,1944,1109,881
SMI,1987,1135,901
snp500,1988,1109,927
Nikkei_225,1677,1174,1062


### Kolmogorov–Smirnov Test

The Kolmogorov–Smirnov (KS) test is a non-parametric statistical test used to determine whether two samples follow the same probability distribution, or whether a sample follows a specific theoretical distribution.



In [10]:
def ks_optimal_split(series, min_ratio=0.1, ratio_target=None, ratio_tol=0.1, nu_tol=0.2):
    x = np.asarray(series.dropna().values, dtype=np.float64)
    n = len(x)
    m = max(30, int(n * min_ratio))
    if n < 3 * m:
        return x[:0], x[:0], x[:0], 0, 0, np.inf, np.array([np.nan, np.nan, np.nan])
    p1 = np.r_[0.0, np.cumsum(x)]
    p2 = np.r_[0.0, np.cumsum(x * x)]
    p3 = np.r_[0.0, np.cumsum(x * x * x)]
    p4 = np.r_[0.0, np.cumsum(x * x * x * x)]

    def nu_seg(a, b):
        l = b - a
        if l < m:
            return np.nan
        s1 = p1[b] - p1[a]
        s2 = p2[b] - p2[a]
        s3 = p3[b] - p3[a]
        s4 = p4[b] - p4[a]
        mu = s1 / l
        ex2 = s2 / l
        ex3 = s3 / l
        ex4 = s4 / l
        c2 = ex2 - mu * mu
        if c2 <= 1e-12:
            return np.nan
        c4 = ex4 - 4 * mu * ex3 + 6 * mu * mu * ex2 - 3 * mu**4
        k = c4 / (c2 * c2) - 3.0
        if not np.isfinite(k):
            return np.nan
        if k <= 1e-8:
            return 200.0
        nu = 4.0 + 6.0 / k
        if not np.isfinite(nu) or nu <= 4.0:
            return np.nan
        return min(nu, 200.0)

    best = None
    for i in range(m, n - 2 * m + 1):
        for j in range(i + m, n - m + 1):
            if ratio_target is not None:
                r = np.array([i, j - i, n - j], dtype=np.float64) / n
                if np.any(np.abs(r - ratio_target) > ratio_tol * ratio_target):
                    continue
            nus = np.array([nu_seg(0, i), nu_seg(i, j), nu_seg(j, n)])
            if np.any(~np.isfinite(nus)):
                continue
            nu_mean = nus.mean()
            if nu_mean <= 0:
                continue
            if np.any(np.abs(nus - nu_mean) > nu_tol * nu_mean):
                continue
            loss = np.abs(nus - nu_mean).sum()
            if best is None or loss < best[0]:
                best = (loss, i, j, nus)

    if best is None:
        i = int(0.6 * n)
        j = int(0.8 * n)
        nus = np.array([nu_seg(0, i), nu_seg(i, j), nu_seg(j, n)])
        loss = np.inf
    else:
        loss, i, j, nus = best

    train = x[:i]
    val = x[i:j]
    test = x[j:]
    return train, val, test, i, j, loss, nus

In [11]:
tmp = {}
for name, s in datasets.items():
    x = s.dropna()
    _, _, _, i, j, _, _ = ks_optimal_split(x)
    n = len(x)
    tmp[name] = (n, i, j)

g = np.mean(np.array([[v[1] / v[0], (v[2] - v[1]) / v[0], (v[0] - v[2]) / v[0]] for v in tmp.values()]), axis=0)
rows = []
for name, s in datasets.items():
    x = s.dropna()
    train, val, test, i, j, _, _ = ks_optimal_split(x, ratio_target=g)
    rows.append({'Dataset': name, 'train_size': len(train), 'val_size': len(val), 'test_size': len(test), 'split_i': i, 'split_j': j})
summary_split = pd.DataFrame(rows)
split_df = summary_split.set_index('Dataset')[['train_size', 'val_size', 'test_size']]
split_df.columns = ['Train', 'Val', 'Test']
summary_split

,Dataset,train_size,val_size,test_size,split_i,split_j
0,Stock returns,730,413,356,730,1143
1,VN30 Index,1971,1096,925,1971,3067
2,VN Index,1964,1103,925,1964,3067
3,DAX_40,1983,1104,972,1983,3087
4,EuroNext_100,2005,1115,979,2005,3120
5,IBEX_35,1815,1362,923,1815,3177
6,KOSPI_index,1944,1109,881,1944,3053
7,SMI,1987,1135,901,1987,3122
8,snp500,1988,1109,927,1988,3097
9,Nikkei_225,1677,1174,1062,1677,2851


In [12]:
summary_split

,Dataset,train_size,val_size,test_size,split_i,split_j
0,Stock returns,730,413,356,730,1143
1,VN30 Index,1971,1096,925,1971,3067
2,VN Index,1964,1103,925,1964,3067
3,DAX_40,1983,1104,972,1983,3087
4,EuroNext_100,2005,1115,979,2005,3120
5,IBEX_35,1815,1362,923,1815,3177
6,KOSPI_index,1944,1109,881,1944,3053
7,SMI,1987,1135,901,1987,3122
8,snp500,1988,1109,927,1988,3097
9,Nikkei_225,1677,1174,1062,1677,2851


## LSTM-GARCH Model Architecture

The LSTM-GARCH hybrid model integrates traditional econometric GARCH volatility modeling with Long Short-Term Memory (LSTM) networks to capture both classical volatility clustering patterns and complex nonlinear dependencies in financial time series.

### Model Framework

The architecture combines two main components: an enhanced GARCH foundation that models volatility clustering and leveraged effects, and an LSTM memory mechanism that captures nonlinear temporal patterns. The integration allows the model to adaptively adjust volatility forecasts based on learned market dynamics.

### Enhanced GARCH Component with HAR Features

The base volatility equation extends the classical GARCH(1,1) formulation with Heterogeneous Autoregressive (HAR) components:

$$\text{base\_var}_t = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2 + \lambda \varepsilon_{t-1}^2 \mathbf{I}(\varepsilon_{t-1} < 0) + \phi_1 RV_{1,t-1} + \phi_5 RV_{5,t-1} + \phi_{20} RV_{20,t-1}$$

where:
- $\omega$ represents the unconditional variance
- $\alpha$ and $\beta$ are standard ARCH and GARCH coefficients  
- $\lambda$ captures leverage effects
- $RV_{1,t} = \varepsilon_{t-1}^2$, $RV_{5,t}$ = 5-day average of squared returns, $RV_{20,t}$ = 20-day average of squared returns
- $\phi_1, \phi_5, \phi_{20}$ are HAR coefficients with normalized constraints

### LSTM Memory Mechanism

The LSTM component processes standardized inputs and applies element-wise corrections:

**Input Features:**
- Standardized shock: $s_t = \frac{\varepsilon_{t-1}}{\sqrt{\sigma_{t-1}^2 + \epsilon}}$
- Log variance: $v_t = \log(\text{base\_var}_t + \epsilon)$

**LSTM Gates (Element-wise Operations):**
$$f_t = \sigma(W_f \odot s_t + U_f \odot v_t + b_f)$$
$$i_t = \sigma(W_i \odot s_t + U_i \odot v_t + b_i)$$
$$\tilde{C}_t = \tanh(W_c \odot s_t + U_c \odot v_t + b_c)$$

where $\odot$ denotes element-wise multiplication.

**Cell State Evolution:**
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**Output Correction:**
$$\text{correction}_t = \tanh(C_t^T v)$$

### Hybrid Integration

The final volatility specification combines GARCH dynamics with multiplicative LSTM adjustments:

$$\sigma_t^2 = \text{base\_var}_t \times (1 + 0.05 \times \tanh(\text{correction}_t))$$

### Statistical Distribution

The model assumes returns follow a Student-t distribution with learnable degrees of freedom:

$$r_t | \mathcal{F}_{t-1} \sim \text{Student-t}(\nu, 0, \sigma_t^2)$$

where $\nu \geq 4$ is constrained to ensure finite fourth moments.

### Optimization Objective

The model is optimized using a composite loss function:

$$\mathcal{L} = \underbrace{\mathbb{E}[\log(\sigma_t^2) + \frac{\varepsilon_t^2}{\sigma_t^2}]}_{\text{QLIKE}} + \underbrace{0.05 \times \mathbb{E}[\text{ReLU}(\varepsilon_t^2 - \sigma_t^2)^2]}_{\text{Spike Penalty}}$$

This loss function combines quasi-maximum likelihood estimation with regularization to prevent extreme volatility spikes.

In [7]:
    class LSTMGARCH(nn.Module):
        def __init__(self, hidden_dim=16):
            super().__init__()
            self.hidden_dim = hidden_dim
            self.raw_omega = nn.Parameter(torch.tensor(-5.0))
            self.raw_alpha = nn.Parameter(torch.tensor(-2.0))
            self.raw_beta = nn.Parameter(torch.tensor(0.0))
            self.raw_lambda = nn.Parameter(torch.tensor(-2.0))
            self.raw_phi1 = nn.Parameter(torch.tensor(-1.0))
            self.raw_phi5 = nn.Parameter(torch.tensor(-1.0))
            self.raw_phi20 = nn.Parameter(torch.tensor(-1.0))
            self.Wf = nn.Parameter(torch.randn(self.hidden_dim))
            self.Uf = nn.Parameter(torch.randn(self.hidden_dim))
            self.bf = nn.Parameter(torch.zeros(self.hidden_dim))
            self.Wi = nn.Parameter(torch.randn(self.hidden_dim))
            self.Ui = nn.Parameter(torch.randn(self.hidden_dim))
            self.bi = nn.Parameter(torch.zeros(self.hidden_dim))
            self.Wc = nn.Parameter(torch.randn(self.hidden_dim))
            self.Uc = nn.Parameter(torch.randn(self.hidden_dim))
            self.bc = nn.Parameter(torch.zeros(self.hidden_dim))
            self.v = nn.Parameter(torch.randn(self.hidden_dim))
            self.w = nn.Parameter(torch.tensor(0.0))
            self.raw_nu = nn.Parameter(torch.tensor(4.0))

        def garch_params(self):
            omega = F.softplus(self.raw_omega)
            alpha = torch.sigmoid(self.raw_alpha)
            beta = torch.sigmoid(self.raw_beta)
            scale = alpha + beta + 1e-6
            alpha = alpha / scale * 0.95
            beta = beta / scale * 0.95
            lambda_ = torch.sigmoid(self.raw_lambda)
            phi1 = torch.sigmoid(self.raw_phi1)
            phi5 = torch.sigmoid(self.raw_phi5)
            phi20 = torch.sigmoid(self.raw_phi20)
            phi_sum = phi1 + phi5 + phi20 + 1e-6
            phi1 = phi1 / phi_sum * 0.6
            phi5 = phi5 / phi_sum * 0.3
            phi20 = phi20 / phi_sum * 0.1
            return omega, alpha, beta, lambda_, phi1, phi5, phi20

        def student_nu(self):
            return torch.clamp(F.softplus(self.raw_nu) + 2, min=4)

        def forward(self, returns):
            omega, alpha, beta, lambda_, phi1, phi5, phi20 = self.garch_params()
            nu = self.student_nu()
            batch_size, T = returns.shape
            sigma2_list = []
            c_t = torch.zeros(batch_size, self.hidden_dim, device=returns.device)
            sigma2_t = returns[:, 0] ** 2 + 1e-6
            sigma2_list.append(sigma2_t)
            squared_returns_history = [returns[:, 0] ** 2]
            for t in range(1, T):
                eps_prev = returns[:, t - 1]
                shock = eps_prev / torch.sqrt(sigma2_t + 1e-8)
                neg = (eps_prev < 0).float()
                leverage = lambda_ * eps_prev ** 2 * neg
                RV1 = eps_prev ** 2
                if len(squared_returns_history) >= 5:
                    RV5 = torch.stack(squared_returns_history[-5:], dim=1).mean(dim=1)
                else:
                    RV5 = torch.stack(squared_returns_history, dim=1).mean(dim=1)
                if len(squared_returns_history) >= 20:
                    RV20 = torch.stack(squared_returns_history[-20:], dim=1).mean(dim=1)
                else:
                    RV20 = torch.stack(squared_returns_history, dim=1).mean(dim=1)
                base_var = omega + alpha * eps_prev ** 2 + beta * sigma2_t + leverage + phi1 * RV1 + phi5 * RV5 + phi20 * RV20
                log_sigma = torch.log(base_var + 1e-8)
                shock = shock.unsqueeze(1)
                log_sigma = log_sigma.unsqueeze(1)
                f_t = torch.sigmoid(self.Wf * shock + self.Uf * log_sigma + self.bf)
                i_t = torch.sigmoid(self.Wi * shock + self.Ui * log_sigma + self.bi)
                c_hat = torch.tanh(self.Wc * shock + self.Uc * log_sigma + self.bc)
                c_t = f_t * c_t + i_t * c_hat
                correction = torch.tanh((c_t * self.v).sum(dim=1))
                sigma2_t = base_var * (1 + 0.05 * torch.tanh(correction))
                sigma2_t = torch.clamp(sigma2_t, min=1e-8)
                sigma2_list.append(sigma2_t)
                squared_returns_history.append(eps_prev ** 2)
            sigma2 = torch.stack(sigma2_list, dim=1)
            eps = returns / torch.sqrt(sigma2)
            term1 = 0.5 * torch.log(sigma2)
            term2 = (nu + 1) / 2 * torch.log(1 + eps ** 2 / ((nu - 2) * sigma2))
            const = torch.lgamma((nu + 1) / 2) - torch.lgamma(nu / 2) - 0.5 * torch.log((nu - 2) * math.pi)
            nll = term1 + term2 - const
            return nll.mean(), sigma2


Training the LSTM - GARCH models

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
seq_len = 60

def create_sequences(data, seq_len):
    xs = []
    for i in range(len(data) - seq_len):
        xs.append(data[i:i+seq_len])
    return np.array(xs, dtype=np.float32)

def compute_metrics(realized_vol, predicted_vol, realized_var, predicted_var):
    mse = np.mean((realized_vol - predicted_vol) ** 2)
    qlike = np.mean(np.log(predicted_var) + realized_var / predicted_var)
    return mse, qlike

def get_ml_predictions_rolling(model, history_returns, test_returns, seq_len):
    preds = []
    used_returns = []
    with torch.no_grad():
        for r in test_returns:
            if len(history_returns) < seq_len:
                history_returns.append(r)
                continue
            seq = torch.tensor(history_returns[-seq_len:], dtype=torch.float32, device=device).unsqueeze(0)
            _, var_seq = model(seq)
            preds.append(var_seq[:, -1].sqrt().cpu().numpy()[0])
            used_returns.append(r)
            history_returns.append(r)
    return np.array(preds), np.array(used_returns)

def train_lstm_garch_for_series(name, series, splits, epochs=100):
    if name not in splits.index:
        return None
    s = series.dropna().values.astype(np.float32)
    train_len = int(splits.loc[name, "Train"])
    val_len = int(splits.loc[name, "Val"])
    test_len = int(splits.loc[name, "Test"])
    if train_len + val_len + test_len > len(s) or train_len <= seq_len or test_len <= seq_len:
        return None
    train = s[:train_len]
    val = s[train_len:train_len + val_len]
    test = s[train_len + val_len:train_len + val_len + test_len]
    train_seq = create_sequences(train, seq_len)
    if len(train_seq) == 0:
        return None
    train_tensor = torch.tensor(train_seq, dtype=torch.float32)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_tensor, batch_size=64, shuffle=True, pin_memory=pin)
    model = LSTMGARCH().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    for _ in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)
            opt.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            opt.step()
    history = list(train)
    ml_vol, returns_eval = get_ml_predictions_rolling(model, history, test.tolist(), seq_len)
    if len(ml_vol) == 0:
        return None
    realized_vol = np.abs(returns_eval)
    realized_var = returns_eval ** 2
    ml_var = ml_vol ** 2
    mse, qlike = compute_metrics(realized_vol, ml_vol, realized_var, ml_var)
    var_calc = var_calculator.RollingVaRCalculator(confidence_level=0.95, min_history=30, refit_frequency=5)
    var_estimates = var_calc.compute_var_rolling_window(returns_eval, ml_vol, confidence_level=0.95)
    valid_mask = (~np.isnan(var_estimates)) & (~np.isnan(returns_eval))
    if not valid_mask.any():
        return None
    returns_valid = returns_eval[valid_mask]
    vol_valid = ml_vol[valid_mask]
    var_valid = var_estimates[valid_mask]
    viol_rate, kupiec_lr, kupiec_p = var_calculator.compute_kupiec_test(returns_valid, var_valid, confidence_level=0.95)
    traffic_light, cum_violations = var_calculator.compute_traffic_light_test(returns_valid, var_valid, confidence_level=0.95)
    lr_ind, p_ind = var_calculator.compute_christoffersen_independence(returns_valid, var_valid, confidence_level=0.95)
    return {
        "Dataset": name,
        "MSE": f"{mse:.6f}",
        "QLIKE": f"{qlike:.6f}",
        "Violation_Rate": f"{viol_rate:.4f}",
        "Kupiec_LR": f"{kupiec_lr:.4f}" if not np.isnan(kupiec_lr) else "N/A",
        "Kupiec_p": f"{kupiec_p:.4f}" if not np.isnan(kupiec_p) else "N/A",
        "LR_Ind": f"{lr_ind:.4f}" if not np.isnan(lr_ind) else "N/A",
        "p_Ind": f"{p_ind:.4f}" if not np.isnan(p_ind) else "N/A",
        "Traffic_Light": traffic_light,
        "Cum_Violations": int(cum_violations) if not np.isnan(cum_violations) else "N/A",
        "Mean_Nu": "N/A",
    }


In [16]:
# Train + test LSTM-GARCH across datasets
results = []
for name, series in datasets.items():
    res = train_lstm_garch_for_series(name, series, split_df, epochs=200)
    if res is not None:
        results.append(res)

if results:
    results_df = pd.DataFrame(results)
    print("\n" + "="*80)
    print("LSTM-GARCH RESULTS")
    print("="*80)
    print(results_df)
    print("="*80)
else:
    print("No results generated (check splits/sequence length).")

NameError: name 'datasets' is not defined

In [35]:
results_df

,Dataset,MSE,QLIKE,Violation_Rate,Kupiec_LR,Kupiec_p,LR_Ind,p_Ind,Traffic_Light,Cum_Violations,Mean_Nu
0,Stock returns,1.353511,1.353553,0.0089,18.0210,0.0000,0.0539,0.8164,Green,3,3.4665
1,VN30 Index,0.022198,-3.699206,0.0000,N/A,N/A,N/A,N/A,Green,0,6.8072
2,VN Index,1.309424,1.279683,0.0129,12.6836,0.0004,0.1046,0.7464,Green,4,5.8786
3,DAX_40,0.926853,1.171321,0.0000,N/A,N/A,N/A,N/A,Green,0,3.5116
4,EuroNext_100,0.740781,0.783983,0.0103,28.4499,0.0000,11.4792,0.0007,Green,5,20.0642
5,IBEX_35,0.903265,1.063701,0.0102,19.0486,0.0000,4.9790,0.0257,Green,3,14.3045
6,KOSPI_index,1.305005,1.599496,0.0133,11.9209,0.0006,0.1081,0.7423,Green,4,4.3246
7,SMI,0.909533,0.843222,0.0156,8.6806,0.0032,4.1605,0.0414,Green,3,5.0712
8,snp500,1.447073,1.180534,0.0154,6.6711,0.0098,4.9343,0.0263,Green,3,4.0872
9,Nikkei_225,1.763598,1.659140,0.0178,6.4706,0.0110,0.1455,0.7029,Green,4,7.2275


In [8]:
data = {
    "Dataset": [
        "Stock returns", "VN30 Index", "VN Index", "DAX_40",
        "EuroNext_100", "IBEX_35", "KOSPI_index", "SMI",
        "snp500", "Nikkei_225"
    ],
    "MSE": [3.645909, 0.000226, 2.156749, 1.794767, 2.043014, 4.350714, 1.603690, 0.972589, 0.702171, 2.522052],
    "QLIKE": [1.791973, -7.476546, 1.430401, 1.298465, 1.072953, 1.772669, 1.695694, 0.667386, 0.624339, 1.785674],
    "Violation_Rate": [0.0506, 0.0401, 0.0676, 0.0159, 0.0162, 0.0083, 0.0554, 0.0441, 0.0364, 0.0205],
    "Kupiec_LR": [0.0018, 0.5989, 1.6614, 4.1630, 17.8908, 19.9838, 0.1581, 0.1758, 0.7107, 4.5489],
    "Kupiec_p": [0.9658, 0.4390, 0.1974, 0.0413, 0.0000, 0.0000, 0.6909, 0.6750, 0.3992, 0.0329]
}

results_df = pd.DataFrame(data)

In [10]:
print(results_df)

         Dataset       MSE     QLIKE  Violation_Rate  Kupiec_LR  Kupiec_p
0  Stock returns  3.645909  1.791973          0.0506     0.0018    0.9658
1     VN30 Index  0.000226 -7.476546          0.0401     0.5989    0.4390
2       VN Index  2.156749  1.430401          0.0676     1.6614    0.1974
3         DAX_40  1.794767  1.298465          0.0159     4.1630    0.0413
4   EuroNext_100  2.043014  1.072953          0.0162    17.8908    0.0000
5        IBEX_35  4.350714  1.772669          0.0083    19.9838    0.0000
6    KOSPI_index  1.603690  1.695694          0.0554     0.1581    0.6909
7            SMI  0.972589  0.667386          0.0441     0.1758    0.6750
8         snp500  0.702171  0.624339          0.0364     0.7107    0.3992
9     Nikkei_225  2.522052  1.785674          0.0205     4.5489    0.0329


Other models

In [9]:

import sys
import importlib
from pathlib import Path

# ==============================================================================
# 6. Run the pipeline
# ==============================================================================
# Ensure project paths are available for imports
workspace_root = Path.cwd()
if not (workspace_root / "ultility").exists():
    workspace_root = workspace_root.parent
model_dir = workspace_root / "model"

for p in [workspace_root, model_dir]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ultility.data_loader
import ultility.metrics
import ultility.models_garch
import ultility.models_transformer
import train_pipeline

importlib.reload(ultility.data_loader)
importlib.reload(ultility.metrics)
importlib.reload(ultility.models_garch)
importlib.reload(ultility.models_transformer)
importlib.reload(train_pipeline)

print("Starting the benchmarking pipeline...")
#results_df = train_pipeline.run_benchmark(datasets, split_df, seq_len=60)

print("\\n" + "="*80)
print("BASELINE MODELS BENCHMARKING RESULTS")
print("="*80)
if "results_df" in globals():
    display(results_df)
else:
    print("results_df is not available. Run previous cells first or uncomment benchmark line.")
print("="*80)


Starting the benchmarking pipeline...
\n================================================================================
BASELINE MODELS BENCHMARKING RESULTS


,Dataset,MSE,QLIKE,Violation_Rate,Kupiec_LR,Kupiec_p
0,Stock returns,3.645909,1.791973,0.0506,0.0018,0.9658
1,VN30 Index,0.000226,-7.476546,0.0401,0.5989,0.4390
2,VN Index,2.156749,1.430401,0.0676,1.6614,0.1974
3,DAX_40,1.794767,1.298465,0.0159,4.1630,0.0413
4,EuroNext_100,2.043014,1.072953,0.0162,17.8908,0.0000
5,IBEX_35,4.350714,1.772669,0.0083,19.9838,0.0000
6,KOSPI_index,1.603690,1.695694,0.0554,0.1581,0.6909
7,SMI,0.972589,0.667386,0.0441,0.1758,0.6750
8,snp500,0.702171,0.624339,0.0364,0.7107,0.3992
9,Nikkei_225,2.522052,1.785674,0.0205,4.5489,0.0329


In [10]:



# Ensure project paths are available for imports
workspace_root = Path.cwd()
if not (workspace_root / "ultility").exists():
    workspace_root = workspace_root.parent
model_dir = workspace_root / "model"

for p in [workspace_root, model_dir]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))


print("Starting the benchmarking pipeline...")
results_df = train_pipeline.run_benchmark(datasets, split_df, seq_len=60)

print("\\n" + "="*80)
print("BASELINE MODELS BENCHMARKING RESULTS")
print("="*80)
display(results_df)
print("="*80)


Starting the benchmarking pipeline...


Processing Datasets:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch [10/80], Train Loss: 0.910029, Val Loss: 0.442500
Epoch [20/80], Train Loss: 0.896495, Val Loss: 0.444019
Epoch [30/80], Train Loss: 0.873511, Val Loss: 0.450424
Epoch [40/80], Train Loss: 0.833010, Val Loss: 0.464840
Epoch [50/80], Train Loss: 0.791941, Val Loss: 0.465928
Epoch [60/80], Train Loss: 0.753487, Val Loss: 0.475552
Epoch [70/80], Train Loss: 0.731028, Val Loss: 0.472403
Epoch [80/80], Train Loss: 0.664656, Val Loss: 0.466464


Processing Datasets:  10%|█         | 1/10 [2:18:26<20:46:00, 8306.74s/it]


KeyboardInterrupt: 

In [ ]:
results_df.to_csv('results.csv')